                # 05 Phase 1 Performance Package Lab

                Purpose: pull latest code, generate Phase 1 report files,
                create the compact performance ZIP for Codex handoff, inspect
                its contents, and record the next action. This replaces any old
                Phase 1 backtest notebook.
                


## 1. Mount Drive And Project Root


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")
Path(PROJECT_ROOT).mkdir(parents=True, exist_ok=True)


## 2. Pull Latest Code


In [ ]:
import subprocess
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/umutergul74/btcusdt_quant_research.git"
GITHUB_BRANCH = "main"
project_path = Path(PROJECT_ROOT)

if (project_path / ".git").exists():
    subprocess.run(["git", "-C", PROJECT_ROOT, "remote", "set-url", "origin", GITHUB_REPO_URL], check=True)
    subprocess.run(["git", "-C", PROJECT_ROOT, "pull", "--ff-only", "origin", GITHUB_BRANCH], check=True)
else:
    visible_items = [p.name for p in project_path.iterdir() if not p.name.startswith(".")]
    if visible_items:
        raise RuntimeError(
            f"{PROJECT_ROOT} exists but is not a git checkout. "
            "Move existing Drive artifacts aside or clone the repository there before continuing. "
            f"Visible items: {visible_items[:10]}"
        )
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, PROJECT_ROOT], check=True)

if f"{PROJECT_ROOT}/src" not in sys.path:
    sys.path.append(f"{PROJECT_ROOT}/src")
print("Repository is ready at:", PROJECT_ROOT)


## 3. Install Dependencies And Stage Helper


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_ROOT}/requirements.txt"], check=True)

from pprint import pprint
from btc_quant.pipeline.stages import run_stage
from btc_quant.pipeline.stage_registry import list_stage_specs

RUN_ID = "debug_colab"

def run_stage_checked(stage_name, config_name=None):
    print(f"\n=== {stage_name} ===")
    result = run_stage(stage_name, config_name=config_name, project_root=PROJECT_ROOT, run_id=RUN_ID)
    pprint(result)
    if result.get("status") == "FAIL":
        raise RuntimeError(f"Stage failed: {stage_name}")
    return result


## 4. Generate Phase 1 Reports


In [ ]:
report_result = run_stage_checked("generate_phase1_reports", config_name="reports")


## 5. Package Phase 1 Performance Bundle


In [ ]:
bundle_result = run_stage_checked("package_phase1_performance", config_name="reports")


## 6. Inspect Bundle Contents


In [ ]:
                from pathlib import Path
                import zipfile

                zip_path = Path(bundle_result["zip_path"])
                print("Bundle:", zip_path)
                with zipfile.ZipFile(zip_path) as zf:
                    names = zf.namelist()
                print("File count:", len(names))
                for name in names:
                    print(name)
                


## 7. Inspect Gate And Readiness Files


In [ ]:
                import json
                from pathlib import Path

                bundle_dir = Path(PROJECT_ROOT) / "reports" / "experiments" / RUN_ID / "phase1_performance_bundle"
                for name in ["decision_report.json", "phase2_readiness.json", "future_oos_readiness.json", "report_consistency_audit.json"]:
                    print("\n---", name, "---")
                    pprint(json.loads((bundle_dir / name).read_text()))
                


## 8. Final Handoff Path


In [ ]:
                print("Download or share this ZIP with Codex for the next iteration:")
                print(bundle_result["zip_path"])
                print("Reminder: the ZIP contains summaries only, not raw data, trained models, or formal backtest results.")
                
